In [ ]:
import ast
import re
import nltk
import random
import numpy as np
import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
import torch.nn as nn
from sklearn.metrics import f1_score
from gensim.models import Word2Vec
from collections import Counter
# Setting as large the xtick and ytick font sizes in graphs

# plt.rcParams['xtick.labelsize'] = 'large'
# plt.rcParams['ytick.labelsize'] = 'large'

In [ ]:
RANDOM_STATE= 42
BATCH_SIZE = 128
LR = 2e-4
EPOCHS = 100
# MAX_LEN
# tf-idf max feature
# TFIDF_MAXFEATURE=60000 # ถ้าเยอะมาก ต้องจำกัด
# MIN_DF=2

# MLP Config
HIDDEN_DIM=300
MAX_LEN = 256

MAX_VOCAB=20000
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

<a id='IMDB'></a>
# IMDB dataset
We retrieve from [Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews?resource=download) the csv file "IMDB Dataset.csv" consisting of 50'000 IMDB movies and TV shows reviews with their positive or negative sentiment classification.

In [ ]:

# 10 target classes (Reuters-10)
TARGET_TOPICS = ['earn', 'acq', 'money-fx', 'grain', 'crude', 'trade', 'interest', 'ship', 'wheat', 'corn']
LABEL2ID = {t: i for i, t in enumerate(TARGET_TOPICS)}
ID2LABEL = {i: t for t, i in LABEL2ID.items()}
NUM_CLASSES = len(TARGET_TOPICS)

# Read CSV
df_raw = pd.read_csv('/kaggle/input/modlewis-train/ModLewis_train.csv')

# Parse topics column (string -> actual list)
def parse_topics(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return x.strip("[]'").replace("'", "").split()
    return []

df_raw['topics_list'] = df_raw['topics'].apply(parse_topics)

# Explode: 1 row per topic, filter for 10 target topics
df_exploded = df_raw.explode('topics_list')
df_exploded = df_exploded[df_exploded['topics_list'].isin(TARGET_TOPICS)].copy()
df_exploded = df_exploded.drop_duplicates(subset=['new_id', 'topics_list'])

# Map topic -> label id (0-9)
df_exploded['label'] = df_exploded['topics_list'].map(LABEL2ID)

# Keep text + label only
df = df_exploded[['text', 'label']].reset_index(drop=True)
print(f'Total samples: {len(df)}')
print(f'Class distribution:\n{df.label.value_counts().rename(index=ID2LABEL)}')
df

We print the basic properties of the DataFrame. In particular, we note that there are no null values in the DataFrame.

In [ ]:
# Labels are already encoded as integers 0-9 above
df

<a id='preprocessing'></a>
# Data preprocessing
First, we use regular expressions to make the following transformations to the reviews:

- remove punctuation marks
- remove HTML tags
- remove URL's
- remove characters which are not letters or digits
- remove successive whitespaces
- convert the text to lower case
- strip whitespaces from the beginning and the end of the reviews

In [ ]:
# Drop rows with NaN text
df = df.dropna(subset=['text']).reset_index(drop=True)
print(f'After dropping NaN: {len(df)} samples')

idx = random.randint(0, len(df)-1)
before_process = df.iloc[idx]['text']

def process(x):
    x = re.sub(r'[,\.!?:()"]', '', x)
    x = re.sub('<.*?>', ' ', x)
    x = re.sub(r'http\S+', ' ', x)
    x = re.sub('[^a-zA-Z0-9]', ' ', x)
    x = re.sub(r'\s+', ' ', x)
    return x.lower().strip()

df['text'] = df['text'].apply(lambda x: process(x))
after_process = df.iloc[idx]['text']
after_process

Next, we remove stopwords from the reviews using the [word_tokenize()](https://www.nltk.org/_modules/nltk/tokenize.html#word_tokenize) function from the [nltk.tokenize]((https://www.nltk.org/api/nltk.tokenize.html) package.

In [ ]:
# Storing in "sw_set" the set of English stopwords provided by nltk
# Defining and applying the function "sw_remove" which remove stopwords from reviews
# Storing in "after_removal" the example of review after removal of the stopwords

sw_set = set(nltk.corpus.stopwords.words('english'))

def tokenize(text):
    return nltk.tokenize.word_tokenize(text)
    
def sw_remove(x):
    words = tokenize(x.lower())
    filtered_list = [word for word in words if word not in sw_set]
    return filtered_list

df['text'] = df['text'].apply(lambda x: sw_remove(x))
after_removal = sw_remove(after_process)
after_removal

<a id='splitting'></a>
# Data splitting and tokenization
We start by splitting our DataFrame into a training and test lists. We use the [train_test_split()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function from the [sklearn.model_selection](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.model_selection) module which allow to perform the splitting randomly with respect to the index of the DataFrame.

In [ ]:
from sklearn.model_selection import train_test_split

train_rev, tmp_rev, train_sent, tmp_sent = train_test_split(
    df['text'], df['label'], test_size=0.1, random_state=RANDOM_STATE, stratify=df['label']
)
test_rev, val_rev, test_sent, val_sent = train_test_split(
    tmp_rev, tmp_sent, test_size=0.5, random_state=RANDOM_STATE, stratify=tmp_sent
)

print('train:', train_rev.shape)
print('test:', test_rev.shape)
print('val:', val_rev.shape)
print('Train label distribution:')
print(train_sent.value_counts().rename(index=ID2LABEL))

#**Text to Vector by word2vec**

In [ ]:
import multiprocessing
print("CPU cores:", multiprocessing.cpu_count())
CPU_CORES=4

In [ ]:
counter = Counter()
for t in train_rev:
    counter.update(t)

vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
for i, (w, _) in enumerate(counter.most_common(MAX_VOCAB - 2), start=2):
    vocab[w] = i

def encode(text):
    ids = [vocab.get(t, vocab[UNK_TOKEN]) for t in text][:MAX_LEN]
    if len(ids) < MAX_LEN:
       ids = ids + [PAD_ID] * (MAX_LEN - len(ids))
    return ids

vocab_size = len(vocab)
vocab_size

In [ ]:
tokenized_train = train_rev.tolist() # pandas Series -> list
vectorize_model = Word2Vec(
    sentences=tokenized_train, 
    vector_size=HIDDEN_DIM, # embedding size
    window=5,
    min_count=1,
    workers=CPU_CORES,
    sg=1 # 0 = CBOW, 1 = Skip-gram
)

print(type(tokenized_train))
print(type(tokenized_train[0]))
print(tokenized_train[0][:10])

In [ ]:
embedding_weight = np.zeros((vocab_size, HIDDEN_DIM), dtype=np.float32)

# ตารางคำศัพท์ → เวกเตอร์
# PAD row (0) = 0 
# UNK row (1) สุ่มเล็กน้อย
rng = np.random.default_rng(RANDOM_STATE)
embedding_weight[UNK_ID] = rng.normal(0, 0.01, size=(HIDDEN_DIM,)).astype(np.float32)

for word, idx in vocab.items():
    if word in (PAD_TOKEN, UNK_TOKEN):
        continue
    if word in vectorize_model.wv:
        embedding_weight[idx] = vectorize_model.wv[word]

In [ ]:
# def masked_mean_pooling(tokens, model): 
#     if MAX_LEN is not None:
#         tokens = tokens[:MAX_LEN]
#     attn_mask = np.ones(len(tokens), dtype=np.float32)

#     if len(tokens) < MAX_LEN:
#         pad_len = MAX_LEN - len(tokens)
#         tokens = tokens + [PAD_ID] * pad_len
#         attn_mask = np.concatenate([attn_mask, np.zeros(pad_len, dtype=np.float32)])

#     # build (T,D) vectors ให้ตรงตำแหน่งกับ mask
#     embedding_matrix = np.zeros((MAX_LEN, HIDDEN_DIM), dtype=np.float32)
#     for i, w in enumerate(tokens):
#         if w in model.wv:
#             embedding_matrix[i] = model.wv[w]

#     denom = attn_mask.sum()
#     if denom == 0:
#         return np.zeros(HIDDEN_DIM, dtype=np.float32)

#     embedding_matrix *= attn_mask[:, None]
#     avg_vectors =(embedding_matrix.sum(axis=0) / denom)
#     return avg_vectors # mean sentence vectors


In [ ]:
# text to vector
y_train = (train_sent).astype(np.int64).to_numpy()
y_val = (val_sent).astype(np.int64).to_numpy()
y_test = (test_sent).astype(np.int64).to_numpy()

X_train = train_rev
X_val = val_rev
X_test = test_rev

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

**# MLP Part**

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class ImdbDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.tolist() # list of list[str]
        self.y = y.astype(np.int64).to_numpy()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        ids = encode(self.X[idx])   # list[int] length MAX_LEN
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.long)

In [ ]:
class MLPClassifier(torch.nn.Module):
    def __init__(self, input_dim, embedding_weight=None, output_dim: int = 2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, HIDDEN_DIM, padding_idx=0)
        if embedding_weight is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_weight, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        self.layers = nn.Sequential(
            nn.Linear(input_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(HIDDEN_DIM, HIDDEN_DIM // 2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(HIDDEN_DIM //2, HIDDEN_DIM // 4),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(HIDDEN_DIM // 4, output_dim)
        )

    def forward(self, input_ids):  # (B,T)
        emb = self.embedding(input_ids)                # (B,T,D)
        mask = (input_ids != PAD_ID).unsqueeze(-1).float()  # (B,T,1)
        sum_vec = (emb * mask).sum(dim=1)              # (B,D)
        count = mask.sum(dim=1).clamp(min=1)           # (B,1)
        mean_vec = sum_vec / count                     # (B,D)
        return self.layers(mean_vec)

In [ ]:
model = MLPClassifier(input_dim=HIDDEN_DIM, embedding_weight=embedding_weight, output_dim=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total = 0.0, 0

    all_preds = []
    all_labels = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)
        total += y.size(0)

        preds = logits.argmax(dim=1)

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    acc = (y_pred == y_true).mean()
    f1 = f1_score(y_true, y_pred, average="macro")  # label 0/1

    return total_loss / total, acc, f1

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total

In [ ]:
def fit(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc*100:.2f}% | "
            f"val_loss={val_loss:.4f} val_acc={val_acc*100:.2f}% val_f1={val_f1:.4f}"
        )

    return history

In [ ]:
def test(model, test_loader, criterion, device):
    test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)
    print(f"TEST | loss={test_loss:.4f} acc={test_acc*100:.2f}% f1={test_f1:.4f}")
    return test_loss, test_acc, test_f1

In [ ]:
train_ds = ImdbDataset(X_train, train_sent)
val_ds   = ImdbDataset(X_val, val_sent)
test_ds  = ImdbDataset(X_test, test_sent)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Train and Test data

In [ ]:
print("vocab_size =", len(vocab))
print(list(vocab.items())[:20])
sample_tokens = train_rev.iloc[0][:30]    
print(sample_tokens)
print([w in vocab for w in sample_tokens[:10]])
print("encoded:", encode(train_rev.iloc[0])[:30])

x, y = next(iter(train_loader))
print(x[0][:30])
print("nonpad:", (x[0] != 0).sum().item())
print("unk:", (x[0] == 1).sum().item())

In [ ]:
# === DEBUG: ตรวจสอบ label ===
print("NUM_CLASSES:", NUM_CLASSES)
print("Model output dim:", model.layers[-1].out_features)
print()

# ตรวจ y values
print("y_train - min:", y_train.min(), "max:", y_train.max(), "unique:", np.unique(y_train))
print("y_val   - min:", y_val.min(), "max:", y_val.max(), "unique:", np.unique(y_val))
print("y_test  - min:", y_test.min(), "max:", y_test.max(), "unique:", np.unique(y_test))
print()

# ตรวจจาก DataLoader โดยตรง
x_batch, y_batch = next(iter(train_loader))
print("Batch y - min:", y_batch.min().item(), "max:", y_batch.max().item())
print("Batch y unique:", y_batch.unique().tolist())
print("Batch x shape:", x_batch.shape, "Batch y shape:", y_batch.shape)

In [ ]:
history = fit(model, train_loader, val_loader, optimizer, criterion, DEVICE, epochs=EPOCHS)

In [ ]:
test_loss, test_acc,test_f1 = test(model, test_loader, criterion, DEVICE)

In [ ]:
@torch.no_grad()
def predict_text(model, text, device):
    model.eval()
    text = process(text)
    text = sw_remove(text)
    encode_text = encode(text)
    x = torch.tensor([encode_text], dtype=torch.long, device=device)
    logits = model(x)
    probs = torch.softmax(logits, dim=1).squeeze(0)

    pred_id = int(torch.argmax(probs).item())
    confidence = float(probs[pred_id].item()) * 100

    label = ID2LABEL[pred_id]
    return label, confidence, probs.detach().cpu().numpy()

In [ ]:
label, conf, probs = predict_text(model, "The company reported strong earnings growth this quarter", DEVICE)
print("result:", label, f"{conf:.2f}%")
print("All probabilities:")
for i, p in enumerate(probs):
    print(f"  {ID2LABEL[i]:10s}: {p*100:.2f}%")